In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:10pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:80px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:2px;}
table.dataframe{font-size:10pt;} 
</style>
"""))

<font size="6" color="green"><b>ch14. 웹 데이터 수집</b></font>
# 1절. Beautifulsoup와 parser
    (정적 웹크롤링, 공공 api 사용)
    
`pip install bs4` 아나콘다를 설치하면 자동 설치되는 패키지에 포함 
<br>
▪ 공식 사이트 : https://www.crummy.com/software/BeautifulSoup/
<br>
▪ Documentation : https://www.crummy.com/software/BeautifulSoup/bs4/doc/

In [7]:
import requests # HTTP요청 처리하는 lib
# file:// == c:
# http://www.
from requests_file import FileAdapter

In [8]:
# 로컬에 있는 파일을 웹요청하듯이 읽어오는 작업
s = requests.Session()
s.mount("file://", FileAdapter()) # fill://로 시작한느 url을 어뎁터가 처리
response = s.get("file:///ai/lecNote/01_python/data/ch14_sample.html") # c:/ai/lecNote/data/ch14_sample.html
response

<Response [200]>

In [9]:
if response:
    print('해당 url에 접근함')
else:
    print('해당 url에 거부됨')

해당 url에 접근함


In [10]:
response.status_code
# 200 : 정상
# 404 : 없는 페이지

200

In [11]:
response.content # html의 바리너리 형식의 내용

b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90 \xec\x95\x88\xec\x9d\x98 \xeb\x82\xb4\xec\x9a\xa9</div>\r\n  <p>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\xb3\xec\x97\x90\xec\x84\x9c \xed\x99\x9c\xec\x9a\xa9\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4</p>\r\n  <div class="contents">\r\n    \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\xa5\xbc \xec\x96\xb4\xeb\x96\xbb\xea\xb2\x8c \xec\x9e\x91\xec\x84\xb1\xed\x95\x98\xeb\x8a\x90\xeb\x83\x90\xec\x97\x90 \xeb\x94\xb0\xeb\x9d\xbc\r\n    <span>\xeb\x8b\xa4\xeb\xa5\xb8<b>\xec\x9a\x94\xec\x86\x8c\xea\xb0\x80 \xeb\xb0\x98\xed\x99\x98</b></span>\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4\r\n  </div>\r\n  <div>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\

In [12]:
print(response.content.decode('utf-8'))

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
</head>
<body>
  <h1 class="greeting css" id="text">Hello, CSS</h1>
  <h1 class="css">Hi, CSS</h1>
  <div id="subject">subject 선택자 안의 내용</div>
  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>
  <div class="contents">
    선택자를 어떻게 작성하느냐에 따라
    <span>다른<b>요소가 반환</b></span>됩니다
  </div>
  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>
</body>
</html>


In [13]:
response.text

'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject 선택자 안의 내용</div>\r\n  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>\r\n  <div class="contents">\r\n    선택자를 어떻게 작성하느냐에 따라\r\n    <span>다른<b>요소가 반환</b></span>됩니다\r\n  </div>\r\n  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>\r\n</body>\r\n</html>'

In [14]:
#html 파싱 객체
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, # == response.content,
                    "html.parser")
# soup.__str__()
# soup

In [15]:
#soup.select_one('선택자') : 해당 선택자 처음 하나 엘리먼트만
el = soup.select_one('h1.greeting.css')
print('el =', el)
print('el.text = ', el.text)
print('el.string = ', el.string)
print('el의 속성들 = ', el.attrs)
print('el의 class속성 = ', el.attrs['class'])
print('el의 class속성 = ', el.attrs.get('class'))
# print('el의 href속성 = ', el.attrs.get['href'])
print('el의 href속성 = ', el.attrs.get('href'))
print('el의 name = ', el.name)

el = <h1 class="greeting css" id="text">Hello, CSS</h1>
el.text =  Hello, CSS
el.string =  Hello, CSS
el의 속성들 =  {'class': ['greeting', 'css'], 'id': 'text'}
el의 class속성 =  ['greeting', 'css']
el의 class속성 =  ['greeting', 'css']
el의 href속성 =  None
el의 name =  h1


In [16]:
# 2. soup.select('선택자') : 해당 선책자 엘리먼트 다 list로
els = soup.select('h1.css')
print('els =>', els)
print('els들의 text', [el.text for el in els])
print('els들의 string', [el.string for el in els])
print('els들의 속성들', [el.attrs for el in els])
print('els들의 class 속성', [el.attrs.get('class') for el in els])

els => [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>]
els들의 text ['Hello, CSS', 'Hi, CSS']
els들의 string ['Hello, CSS', 'Hi, CSS']
els들의 속성들 [{'class': ['greeting', 'css'], 'id': 'text'}, {'class': ['css']}]
els들의 class 속성 [['greeting', 'css'], ['css']]


In [17]:
#3. soup.find(태그, 속성) vs soup.select_one('선택자') : 해당 속성을 갖은 태그. 처음 하나만
print('select_one :', soup.select_one('h1.css'))
print('find       :', soup.find('h1', {'class':'css'}))
print('find       :', soup.find('h1', class_='css'))
print()
print('select_one :', soup.select_one('h1#text'))
print('select_one :', soup.select_one('h1', {'id':'text'}))

select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>
find       : <h1 class="greeting css" id="text">Hello, CSS</h1>
find       : <h1 class="greeting css" id="text">Hello, CSS</h1>

select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>
select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>


In [18]:
# 4. soup.find_all(태그, 속성) vs soup.select_one('선택자') : 해당 엘리먼트 다 list로
print('모든 h1.css와 span태그 :', soup.select('h1.css, span'))
print('모든 h1.css와 span태그 :', soup.find_all(['h1'], class_=('css') + soup.find_all('span'))

SyntaxError: incomplete input (4136789065.py, line 3)

In [19]:
# 없는 엘리먼트 찾기
print('find_all(빈list) :', soup.find_all('a'))
print('find(None) :', soup.find('a'))
print('select(빈list) :', soup.select('a'))
print('select_one(None) :', soup.select_one('a'))

find_all(빈list) : []
find(None) : None
select(빈list) : []
select_one(None) : None


# 2절. 정적 웹 데이터 수집(정적 웹크롤링)
## 2.1 BeautifulSoup 모듈을 활용한 html 웹 데이터 수집
### 1) 환율 정보 가져오기(네이버증권>시장지표)

- https://finance.naver.com/marketindex/
    * 크롤링 허용범위는 사이트마다 ~/robots.txt에서 확인할 수 있음
        - Allow : 크롤링 허용가능한 폴더
        - Disallow : 크롤링 제한 폴더

In [20]:
# soup 객체 생성 방법 1
import requests
from bs4 import BeautifulSoup
url ='https://finance.naver.com/marketindex/'
requests = requests.get(url)
#requests #Response
print(response.status_code)
# response.text # response.content
soup = BeautifulSoup(response.text, 'html.parser')

200


In [21]:
# soup 객체 생성 방법 2
from urllib.request import urlopen
url = 'https://finance.naver.com/marketindex/'
response = urlopen(url)
# response
print(response.status)
# print(response.read().decode('cp949'))
soup = BeautifulSoup(response, 'html.parser')

200


In [22]:
p = '1,470.70'
float(p.replace(',', '')) # 방법 1
float(''.join(p.split(','))) # 방법2

1470.7

In [23]:
# div.head_info 밑의 span.value (find계열)
prices = []
headinfos = soup.find_all('div', class_='head_info')
for headinfo in headinfos:
    # print(headinfos)
    price = headinfo.find('span', class_='value')
    prices.append(float(''.join(price.text.split(','))))
print(prices)

[1422.5, 892.52, 1639.22, 210.86, 159.23, 1.1541, 1.3508, 99.89, 83.27, 1863.53, 4467.5, 200006.41]


In [24]:
# span.value (find 계열)
price_els = soup.find_all('span', class_='value')
prices = [round(float(price.text.replace(',', '')), 1) for price in price_els]
print(prices)

[1422.5, 892.5, 1639.2, 210.9, 159.2, 1.2, 1.4, 99.9, 83.3, 1863.5, 4467.5, 200006.4]


In [25]:
# div.head_info 밑의 span.value
price_els = soup.select('div.head_info > span.value')
# price_els

In [26]:
title_els = soup.select('h3.h_lst >span.blind')
# title_els

In [27]:
# 단위들 : div.head_info > span > span.blind
unit_els = soup.select('div.head_info > span > span.blind')
len(unit_els)
units = [unit_el.string for unit_el in unit_els]
units.insert(7,'')
units

['원', '원', '원', '원', '엔', '달러', '달러', '', '달러', '원', '달러', '원']

In [28]:
# 상승 / 하락 div.head_info > span.blind
trend_els = soup.select('div.head_info > span.blind')
trend_els

[<span class="blind">상승</span>,
 <span class="blind">상승</span>,
 <span class="blind">상승</span>,
 <span class="blind">상승</span>,
 <span class="blind">하락</span>,
 <span class="blind">보합</span>,
 <span class="blind">상승</span>,
 <span class="blind">상승</span>,
 <span class="blind">상승</span>,
 <span class="blind">하락</span>,
 <span class="blind">상승</span>,
 <span class="blind">하락</span>]

In [29]:
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    print("{} : {}{} - {}".format(title.text, price.text, unit, trend.text))

미국 USD : 1,422.50원 - 상승
일본 JPY(100엔) : 892.52원 - 상승
유럽연합 EUR : 1,639.22원 - 상승
중국 CNY : 210.86원 - 상승
달러/일본 엔 : 159.2300엔 - 하락
유로/달러 : 1.1541달러 - 보합
영국 파운드/달러 : 1.3508달러 - 상승
달러인덱스 : 99.8900 - 상승
WTI : 83.27달러 - 상승
휘발유 : 1863.53원 - 하락
국제 금 : 4467.5달러 - 상승
국내 금 : 200006.41원 - 하락


In [30]:
import pandas as pd
data = []
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    data.append({'title' : title.text,
                'price' : float(price.text.replace(',', '')),
                'unit' : unit,
                'trend' : trend.text})
pd.DataFrame(data) #.to_csv('data/file.csv', index=false)

,title,price,unit,trend
0,미국 USD,1422.5000,원,상승
1,일본 JPY(100엔),892.5200,원,상승
2,유럽연합 EUR,1639.2200,원,상승
3,중국 CNY,210.8600,원,상승
4,달러/일본 엔,159.2300,엔,하락
5,유로/달러,1.1541,달러,보합
6,영국 파운드/달러,1.3508,달러,상승
7,달러인덱스,99.8900,,상승
8,WTI,83.2700,달러,상승
9,휘발유,1863.5300,원,하락


In [31]:
data = []
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    data.append([title.text, float(price.text.replace(',', '')), unit, trend.text])
pd.DataFrame(data, columns=['title', 'price', 'unit', 'trend'])

,title,price,unit,trend
0,미국 USD,1422.5000,원,상승
1,일본 JPY(100엔),892.5200,원,상승
2,유럽연합 EUR,1639.2200,원,상승
3,중국 CNY,210.8600,원,상승
4,달러/일본 엔,159.2300,엔,하락
5,유로/달러,1.1541,달러,보합
6,영국 파운드/달러,1.3508,달러,상승
7,달러인덱스,99.8900,,상승
8,WTI,83.2700,달러,상승
9,휘발유,1863.5300,원,하락


### 2) 이번주 로또번호 출력
- 방법 2에서 user Agent를 추가하여 soup생성
- https://search.daum.net/search?w=tot&DA=YZR&t__nil_searchbox=btn&q=lotto (다음에서 lotto검색)
```
1236(2026.08.08 추첨)
당첨번호 [12, 18,21, 29, 34, 38]
보너스 10
```

In [32]:
# 방법1
import requests
from bs4 import BeautifulSoup
url = 'https://search.daum.net/search?w=tot&DA=YZR&t__nil_searchbox=btn&q=lotto'
response = requests.get(url)
print('response의 상태 :' , response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')
# soup

response의 상태 : 200


In [33]:
# 방법2
from urllib.request import urlopen, Request
headers = {'User-Agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
request = Request(url, headers =headers)

response = urlopen(request)
print('response의 상태 :', response.status)
soup = BeautifulSoup(response, 'html.parser')
# soup

response의 상태 : 200


In [34]:
# 1236회(2026.08.08 추첨)
# 당첨번호 [12, 18,21, 29, 34, 38]
# 보너스 10
times = soup.select_one('div.prize span.f_red').text
date = soup.select_one('div.prize > span.date').text
title1 = soup.select_one('div.prize > strong').text[-4:]
lotto_numbers = soup.select('div.lottonum > span.ball:nth-last-child(n+3)')
# lotto_numbers = soup.select('div.lottonum > span.ball')[2]
title2 = soup.select_one('div.lottonum span.screen_out').text
lotto_bonus = soup.select_one('div.lottonum > span.bg_ball1').text
print(times, date)
print(title1, [int(numbers.text) for numbers in lotto_numbers])
print(title2, lotto_bonus)

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 10


In [35]:
# 위의 select계열 함수를 find계열함수로 변경하여 구현해 보기
prize = soup.find('div', class_ = 'prize')
times = prize.find('span', class_ = 'f_red').text
date = prize.find('span', class_ = 'date').text
title1 = prize.find('strong').text[-4:]

lottonum = soup.find('div', class_ = 'lottonum')
Lotto_numbers = lottonum.find_all('span', class_='ball')[:-2]

                   
print(times, date)
print(title1, [int(numbers.text) for numbers in lotto_numbers])
print(title2, lotto_bonus)

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 10


### 3) DAUM 뉴스 검색 리스트
```
no title href
0 타이틀1 http://~~
1 타이틀2 http://~~
2 타이틀3 http://~~
```

In [36]:
# 방법1
import requests
from bs4 import BeautifulSoup
word = '국립중앙박물관'
url = f'https://search.daum.net/search?w=news&q={word}&enc=utf8&cluster=y&cluster_page=1&DA=DNS'
print(url)
response = requests.get(url)
print(response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')

https://search.daum.net/search?w=news&q=국립중앙박물관&enc=utf8&cluster=y&cluster_page=1&DA=DNS
200


In [37]:
# 방법2
from urllib.request import urlopen, Request
from urllib.parse import quote
word = quote('금시세')
url = f'https://search.daum.net/search?w=news&q={word}'
print(url)
headers = {'User-Agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
# request = Request(url, headers=headers)
request =Request(url)
request.add_header('User-Agent', 
                   'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36')
response = urlopen(request)
print(response.status)
soup =BeautifulSoup(response, 'html.parser')
# soup

https://search.daum.net/search?w=news&q=%EA%B8%88%EC%8B%9C%EC%84%B8
200


In [38]:
items_find_list = [] # 검색한 결과를 담을 dict 리스트
items_el = soup.select('div.item-title > strong.tit-g.clamp-g > a')
for idx, item in enumerate(items_el):
    # print(idx, item)
    items_find_list.append({'no':idx + 1,
                           'title':item.text,
                           'link': item.attrs.get('href')})
import pandas as pd
pd.DataFrame(items_find_list)

,no,title,link
0,1,금시세(금값) 10일,http://v.daum.net/v/20260810091128225
1,2,오늘 금시세(금값)는?,http://v.daum.net/v/20260806100049495
2,3,"[오늘 금시세]금값, 국내·국제 모두 상승세",http://v.daum.net/v/20260813092049615
3,4,"오늘 금값, 8월 4일 한국거래소 금 시세 전 거래일과 비교",http://v.daum.net/v/20260804165453141
4,5,"[금값시세] 8월 13일, 18K 14K 백금 은 가격 한눈에",http://v.daum.net/v/20260813113111643
5,6,금시세(금값) 27일,http://v.daum.net/v/20260727100640129
6,7,"[오늘 금시세]저가 매수세 유입에 금값, 반등…4천달러선 지지 확인",http://v.daum.net/v/20260722092117388
7,8,"[오늘 금시세] Fed 긴축 전망에 금값, 4천달러선 하회",http://v.daum.net/v/20260720092305278
8,9,[EBN 데이터센터] 국제 금값 장중 약세...국내 금시세도 18만원대로 밀려,http://v.daum.net/v/20260724155251980
9,10,금시세(금값) 24일,http://v.daum.net/v/20260724102307514


In [39]:
items_find_list = [] # 검색한 결과를 담을 2차원 리스트
items_el = soup.select('div.item-title > strong.tit-g.clamp-g > a')
for idx, item in enumerate(items_el):
    items_find_list.append([idx, item.text, item.attrs.get('href')])
pd.DataFrame(items_find_list, columns=['순번', '기사제목', '링크'])

,순번,기사제목,링크
0,0,금시세(금값) 10일,http://v.daum.net/v/20260810091128225
1,1,오늘 금시세(금값)는?,http://v.daum.net/v/20260806100049495
2,2,"[오늘 금시세]금값, 국내·국제 모두 상승세",http://v.daum.net/v/20260813092049615
3,3,"오늘 금값, 8월 4일 한국거래소 금 시세 전 거래일과 비교",http://v.daum.net/v/20260804165453141
4,4,"[금값시세] 8월 13일, 18K 14K 백금 은 가격 한눈에",http://v.daum.net/v/20260813113111643
5,5,금시세(금값) 27일,http://v.daum.net/v/20260727100640129
6,6,"[오늘 금시세]저가 매수세 유입에 금값, 반등…4천달러선 지지 확인",http://v.daum.net/v/20260722092117388
7,7,"[오늘 금시세] Fed 긴축 전망에 금값, 4천달러선 하회",http://v.daum.net/v/20260720092305278
8,8,[EBN 데이터센터] 국제 금값 장중 약세...국내 금시세도 18만원대로 밀려,http://v.daum.net/v/20260724155251980
9,9,금시세(금값) 24일,http://v.daum.net/v/20260724102307514


In [40]:
# 다음 뉴스 검색 함수(원하는 키워드, 원하는 페이지 수 입력)
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def collect_list(keyword, page):
    'keyword로 해당 page에 검색한 결과 dict list return'
    # url = f'https://search.daum.net/search?w=news&q={keyword}&enc=utf8&cluster=y&cluster_page=1&DA=PGD&p={page}'
    url = f'https://search.daum.net/search?w=news&enc=utf8&cluster=y&cluster_page=1&DA=PGD'
    params = {'q':keyword, 'p':page}
    response = requests.get(url, params=params)
    soup = BeautifulSoup(response.text, 'html.parser')
    items_find_list = []
    items_el = soup.select('div.item-title > strong.tit-g > a')
    for idx, item in enumerate(items_el):
        items_find_list.append({'no':(page-1)*10 + idx,
                               'title':item.text,
                               'link':item.attrs.get('href')})
    return items_find_list

In [41]:
collect_list('국립중앙박물관', 2)

[{'no': 10,
  'title': ' ‘데뷔 10주년’ 블랙핑크, 국립중앙박물관과 협업…문화유산 디테일 ',
  'link': 'http://v.daum.net/v/20260806151129213'},
 {'no': 11,
  'title': ' 블랙핑크, 국립중앙박물관서 데뷔 10주년 팬 이벤트…4인 완전체 참석 ',
  'link': 'http://v.daum.net/v/20260807172052096'},
 {'no': 12,
  'title': ' “오늘은 학생이에요” 국립중앙박물관서 ‘박물관 교사 연수’ ',
  'link': 'http://v.daum.net/v/20260805110514894'},
 {'no': 13,
  'title': ' 유홍준 관장 “국립중앙박물관 제2관 지으면 일제강점기실 신설” ',
  'link': 'http://v.daum.net/v/20260730132248376'},
 {'no': 14,
  'title': " 국립중앙박물관, 특별전 '우리들의 밥상' 연계 강연·토론회 ",
  'link': 'http://v.daum.net/v/20260728082137519'},
 {'no': 15,
  'title': ' 현장 체험학습 어떻게 할까…국립중앙박물관 교사 연수 ',
  'link': 'http://v.daum.net/v/20260805101554300'},
 {'no': 16,
  'title': ' [포토] 새 단장 마친 국립중앙박물관 대한제국실 ',
  'link': 'http://v.daum.net/v/20260730154707887'},
 {'no': 17,
  'title': ' 국중박이 내놓은 한상차림…"취향따라 달라질 각자의 경험" ',
  'link': 'http://v.daum.net/v/20260812185653026'},
 {'no': 18,
  'title': " 국립중앙박물관, '태국미술명품전' 닷새간 무료 개방 ",
  'link': 'http://v.daum.net/v/2026072

In [42]:
result = [] # 해당 키워드 원하는 페이지 수만큼 검색한 결과를 담을 변수 dict 리스트
pages = 3
for page in range(1, pages+1):
    print(f'=={page} 페이지 수집 중 ==')
    item_result = collect_list('북한산', page)
    result.extend(item_result)
    time.sleep(3)
pd.DataFrame(result)

==1 페이지 수집 중 ==
==2 페이지 수집 중 ==
==3 페이지 수집 중 ==


,no,title,link
0,0,"노후 데크·난간 싹 정비…성북구, '북한산 자락길' 전면 개방",http://v.daum.net/v/20260813100649911
1,1,"[지금 구청은] 성북구, 노후 ‘북한산 자락길’ 보수정비 완료…전면 개방",http://v.daum.net/v/20260813113857054
2,2,"도심 속에 솟아오른 거대한 화강암, 북한산 [두시기행문]",http://v.daum.net/v/20260726124409524
3,3,"""북한산 자락의 이색 휴양지""…안토, 부에노 비치클럽 정식 오픈",http://v.daum.net/v/20260805092529730
4,4,"젤렌스키 ""러, 우크라 공격에 북한산 탄도미사일 사용""",http://v.daum.net/v/20260811164010996
5,5,"젤렌스키 ""러시아, 북한산 탄도미사일 사용""",http://v.daum.net/v/20260811170900267
6,6,“북한산 자락에서 해변의 정취를”,http://v.daum.net/v/20260805105710446
7,7,"“여기는 가봐야지”…강북구 북한산 체험형 숲속쉼터, ‘서울 에디션25’ 최종 선정",http://v.daum.net/v/20260810102609758
8,8,생후 18개월 아기까지 희생…우크라이나 민가 덮친 ‘북한산’ 미사일,http://v.daum.net/v/20260804151931476
9,9,"북한산 한가운데 비치클럽이? 리조트 안토, '부에노 비치클럽' 개장",http://v.daum.net/v/20260805092457716


In [43]:
result

[{'no': 0,
  'title': " 노후 데크·난간 싹 정비…성북구, '북한산 자락길' 전면 개방 ",
  'link': 'http://v.daum.net/v/20260813100649911'},
 {'no': 1,
  'title': ' [지금 구청은] 성북구, 노후 ‘북한산 자락길’ 보수정비 완료…전면 개방 ',
  'link': 'http://v.daum.net/v/20260813113857054'},
 {'no': 2,
  'title': ' 도심 속에 솟아오른 거대한 화강암, 북한산 [두시기행문] ',
  'link': 'http://v.daum.net/v/20260726124409524'},
 {'no': 3,
  'title': ' "북한산 자락의 이색 휴양지"…안토, 부에노 비치클럽 정식 오픈 ',
  'link': 'http://v.daum.net/v/20260805092529730'},
 {'no': 4,
  'title': ' 젤렌스키 "러, 우크라 공격에 북한산 탄도미사일 사용" ',
  'link': 'http://v.daum.net/v/20260811164010996'},
 {'no': 5,
  'title': ' 젤렌스키 "러시아, 북한산 탄도미사일 사용" ',
  'link': 'http://v.daum.net/v/20260811170900267'},
 {'no': 6,
  'title': ' “북한산 자락에서 해변의 정취를” ',
  'link': 'http://v.daum.net/v/20260805105710446'},
 {'no': 7,
  'title': ' “여기는 가봐야지”…강북구 북한산 체험형 숲속쉼터, ‘서울 에디션25’ 최종 선정 ',
  'link': 'http://v.daum.net/v/20260810102609758'},
 {'no': 8,
  'title': ' 생후 18개월 아기까지 희생…우크라이나 민가 덮친 ‘북한산’ 미사일 ',
  'link': 'http://v.daum.net/v/2026080

In [44]:
keywords = ['고윤정', '드라마']
pages = 3
result0 = [] #keyword[0] 1~n페이지까지 검색한 결과를 도출 dict list
result1 = [] # keyword[1] 1~n페이지까지 검색한 결과를 도출 dict list
for i, keyword in enumerate(keywords):
    print(f'= = {i+1}번째 검색어 {keyword} 검색 결과 수집({pages}페이지) 중입니다. = =')
    for page in range(1, pages+1):
        if i==0:
            result0.extend(collect_list(keyword, page))
        else:
            result1.extend(collect_list(keyword, page))
        time.sleep(3)

= = 1번째 검색어 고윤정 검색 결과 수집(3페이지) 중입니다. = =
= = 2번째 검색어 드라마 검색 결과 수집(3페이지) 중입니다. = =


In [45]:
result0_df = pd.DataFrame(result0)
result1_df = pd.DataFrame(result0)
result0_df.sample()

,no,title,link
18,18,"""고윤정 덕분에""...들썩이는 '이 회사'",http://v.daum.net/v/20260725080610898


In [46]:
result1_df.head()

,no,title,link
0,0,"“와 레전드” 고윤정, 청룡서 리즈 찍었다… 등근육 자랑까지 [IS하이컷]",http://v.daum.net/v/20260802181752582
1,1,"고윤정, 이 미모에 성난 등근육이라니…""무슨 운동 하길래?""",http://v.daum.net/v/20260804040150778
2,2,"고윤정, 압도적인 분위기...‘청룡 레드카펫 장악’[케이팝 헌터스]",http://v.daum.net/v/20260811110354329
3,3,"고윤정 프랑스 니스 여행 패션, 선글라스부터 티셔츠까지 총정리",http://v.daum.net/v/20260806053128482
4,4,"고윤정, 이병헌 만난다더니…""승마, 액션 훈련 준비 중"" 연기 스펙트럼 또 넓힌다",http://v.daum.net/v/20260713160107321


In [47]:
result0_df.to_csv(f'Data/ch14_{keywords[0]}.csv', index=False, encoding='cp949')
result1_df.to_csv(f'Data/ch14_{keywords[1]}.csv', index=False, encoding='cp949')

### 4) User-Agent를 추가하여 크롤링
- request.get(url), urlopen(url)함수를 사용하면 크롤링이 막혀있는 사이트
- 방법2에서 User-Agent를 추가하여 크롤링

- Melon.com
- https://www.melon.com/robots.txt 에서 일부 경로는 User-Agent에 봇이 지정

In [48]:
# 방법 1
import requests
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart'
melonResponse = requests.get(url)
print(melonResponse.status_code)
soup = BeautifulSoup(melonResponse.text, 'html.parser')
soup

406


In [49]:
# 방법2
from urllib.request import urlopen
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart'
# melonResponse = urlopen(url) # HTTP Error 406: Not Acceptable 에러뜸.

In [50]:
# User-Agent를 추가하여 방법2
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
headers = {'user-agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
# melonpage = Request(url, headers=headers)
melonpage = Request(url)
melonpage.add_header('user-agent',
                    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36')
melonResponse = urlopen(melonpage)
print(melonResponse.status)
soup = BeautifulSoup(melonResponse, 'html.parser')
# soup

200


In [51]:
# User-Agent를 추가하여 방법1
import requests
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
headers = {'user-agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
melonResponse = requests.get(url, headers=headers)
print(melonResponse.status_code)
soup = BeautifulSoup(melonResponse.text, 'html.parser')
# soup

200


In [52]:
# 1위 : LOVE ATTACK | RESCENE(리센느)의 {}
# 순위, 곡명, 가수, 가수페이지
rank_els = soup.select('div.wrap.t_center > span.rank')[1:] #맨앞에 순위가 있어서 1번부터 시작
ranks = [rank.text for rank in rank_els]

title_els = soup.select('div.ellipsis.rank01 > span > a')
titles = [title.text for title in title_els]

singer_els = soup.select('span.checkEllipsis') # 가수가 복수명인 순위가 있어서,
singers = [singer.text.replace('\xa0', '') for singer in singer_els]

# 가수 복수
# link_els = soup.select('span.checkEllipsis > a')
# links = ['https://www.melon.com/'+link.attrs.get('href') for link in link_els]

links = []
for singer_el in singer_els:
    # print(singer_el)
    singer_link = 'https://www.melon.com/' + singer_el.find('a').attrs.get('href')
    links.append(singer_link)
    
# for idx, (title, singer, link) in enumerate(zip(titles, singers, links)):
#     print("{}위, {} | {}".format(idx+1, title, singer))
melon_chat_list = []
for rank, title, singer, link in zip(ranks, titles, singers, links):
#     print(f'{rank}위 {title} | {singer}')
    melon_chat_list.append({
        '순위':rank,
        '곡명':title,
        '가수':singer,
        '가수페이지':link
    })
pd.DataFrame(melon_chat_list)

,순위,곡명,가수,가수페이지
0,1,LOVE ATTACK,RESCENE(리센느),https://www.melon.com//artist/detail.htm?artis...
1,2,갑자기,아이오아이(I.O.I),https://www.melon.com//artist/detail.htm?artis...
2,3,REDRED,CORTIS(코르티스),https://www.melon.com//artist/detail.htm?artis...
3,4,LEMONADE,aespa,https://www.melon.com//artist/detail.htm?artis...
4,5,Pretty Girl,RESCENE(리센느),https://www.melon.com//artist/detail.htm?artis...
...,...,...,...,...
95,96,OVERDRIVE,TWS(투어스),https://www.melon.com//artist/detail.htm?artis...
96,97,순간을 영원처럼,임영웅,https://www.melon.com//artist/detail.htm?artis...
97,98,Hooligan,방탄소년단,https://www.melon.com//artist/detail.htm?artis...
98,99,우리들의 블루스,임영웅,https://www.melon.com//artist/detail.htm?artis...


### 5) 네이버 지식인 검색(open API 사용x)
- 특정 Keyword를 특정 페이지 수만큼

In [53]:
# 방법 1
from requests import get
from bs4 import BeautifulSoup
keyword = '참외'
url =f'https://kin.naver.com/search/list.naver?query={keyword}'
print(url)
response = get(url)
print(response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')

https://kin.naver.com/search/list.naver?query=참외
200


In [54]:
# 방법 2
from urllib.request import urlopen
from bs4 import BeautifulSoup
from urllib.parse import quote
keyword = quote('참외')
url =f'https://kin.naver.com/search/list.naver?query={keyword}'
print(url)
response = urlopen(url)
print(response.status)
soup = BeautifulSoup(response, 'html.parser')

https://kin.naver.com/search/list.naver?query=%EC%B0%B8%EC%99%B8
200


In [55]:
# keyword를 원하는 페이지 수만큼
# 방법 1
from requests import get
from bs4 import BeautifulSoup
keyword = '참외'
pages = 2
items_list = [] # 크롤링한 데이터를 담을 list
for page in range(1, pages+1):
    url = f'https://kin.naver.com/search/list.naver?query={keyword}&page={page}'
#     url = 'https://kin.naver.com/search/list.naver'
#     params = {'query':keyword, 'page':page}
#     response = get(url, params=params)
    response = get(url)
#     print(response.status_code)
    soup = BeautifulSoup(response.text, 'html.parser')
    #  글제목, link
    dt_els = soup.find_all('dt')
    # print(len(dt_els))
    for dt_el in dt_els:
        item = dt_el.find('a') #, class_=['_nclicks:qna.txt', '_searchListTitleAnchor']
        items_list.append({'title' : item.text,
                          'link':item.attrs.get('href')})
df = pd.DataFrame(items_list)

In [56]:
df.head()

,title,link
0,참외짱아찌 상태 봐주세요,https://kin.naver.com/qna/detail.naver?dirId=8...
1,참외껍질로 만든 가죽제품은 수지타산이 맞....,https://kin.naver.com/qna/detail.naver?dirId=4...
2,애플참외 수확시기,https://kin.naver.com/qna/detail.naver?dirId=8...
3,참외 속이 이상해요. 먹어도 되나요?,https://kin.naver.com/qna/detail.naver?dirId=8...
4,참외 알레르기가 있으면 다른 과일도 조심해....,https://kin.naver.com/qna/detail.naver?dirId=7...


## 2.2 open API사용 : json 웹데이터 수집
### 1) 네이버 지식으로 검색(open API사용 o)

- 네이버 개발자 센터에서 어플리케이션을 등록(id, pw) => NAVER API HUB

In [57]:
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.


In [58]:
# 환경변수를 쓰기 위한 패키지 : doten
from dotenv import load_dotenv
import os
load_dotenv(
# 안쓰면,기본값 → dotenv_path='.env'
)
print(os.getenv('CLIENT_ID')[:3])
print(os.getenv('CLIENT_SECRET')[:3])

QcX
HJe


In [2]:
# 방법 1
import os
import requests
import json
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = "젤라또"
url = f"https://openapi.naver.com/v1/search/kin.json" # JSON 결과
params = {'query':encText, 'display':20}
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
response = requests.get(url, params=params, headers=headers)

items = response.json()['items']
items_list = []
for item in items:
    title = item.get('title').replace('<b>', '').replace('</b>','')
    link = item['link']
    description = item['description'].replace('<b>', '').replace('</b>','')
    items_list.append([title, link, description])
pd.DataFrame(items_list, columns=['title', 'link', 'description']).sample()

,title,link,description
13,"수박소르베 밀키스무디 vs 팥빙 젤라또 파르페, 어떤 게 더 ....",https://kin.naver.com/qna/detail.naver?dirId=8...,메가커피에서 여름 신메뉴로 나온 수박소르베 밀키스무디와 팥빙 젤라또 파르페 중에 고...


In [4]:
# 방법 2 # 네이버 검색 API 예제 - 블로그 검색
import os
import sys
import urllib.request
import json
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = urllib.parse.quote("냉장고")
url = f"https://openapi.naver.com/v1/search/kin.json?query={encText}" # JSON 결과
# url = "https://openapi.naver.com/v1/search/blog.xml?query=" + encText # XML 결과
# request = urllib.request.Request(url)
# request.add_header("X-Naver-Client-Id",client_id)
# request.add_header("X-Naver-Client-Secret",client_secret)
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
request = urllib.request.Request(url, headers=headers)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    # print(response_body.decode('utf-8'))
else:
    print("Error Code:" + rescode)
    
data = json.loads(response_body) # 문자를 json형태로 변환
print('data의 응답들 :', data.keys())
print('검색결과 갯수 :', len(data['items']))

items = data['items']
items_list = []
for item in items:
    title = item.get('title').replace('<b>', '').replace('</b>','')
    link = item['link']
    description = item['description'].replace('<b>', '').replace('</b>','')
    items_list.append([title, link, description])
pd.DataFrame(items_list, columns=['title', 'link', 'description']).sample()

data의 응답들 : dict_keys(['lastBuildDate', 'total', 'start', 'display', 'items'])
검색결과 갯수 : 10


,title,link,description
8,새 냉장고 소음,https://kin.naver.com/qna/detail.naver?dirId=5...,삼성 냉장고를 어제 오전 설치하자마자 옅은 고주파음이 계속 들립니다. 서비스기사님이...


### NAVER API HUB

In [61]:
# 방법 1
import os
import requests
import json
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
client_id = os.getenv('NEW_CLIENT_ID')
client_secret = os.getenv('NEW_CLIENT_SECRET')
encText = "젤라또"
url = f"https://naverapihub.apigw.ntruss.com/search/v1/kin" # JSON 결과
params = {'query':encText, 'display':20 }
headers = {
    'X-NCP-APIGW-API-KEY-ID': client_id,
    'X-NCP-APIGW-API-KEY':client_secret
}
response = requests.get(url, params=params, headers=headers)

items = response                                      .json()['items']
items_list = []
for item in items:
    title = item.get('title').replace('<b>', '').replace('</b>','')
    link = item['link']
    description = item['description'].replace('<b>', '').replace('</b>','')
    items_list.append([title, link, description])
pd.DataFrame(items_list, columns=['title', 'link', 'description']).sample()

,title,link,description
10,"메가커피 여름 신메뉴, 팥빙 젤라또 파르페 맛 어땠나요?",https://kin.naver.com/qna/detail.naver?dirId=8...,"팥빙 젤라또 파르페를 먹어봤는데, 생각보다 양이 많고 가성비가 좋더라고요. 그런데 ..."


### Quiz _ 네이버open API를 이용하여 원하는 query 이미지 100건의 데이터를 'data/img_list.csv'파일로
- title

In [25]:
def get_image_list(query):
    'query로 검색한 이미지 정보(제목, 링크,썸네일, size) 100건 데이터 프레임을 return(방법1)'
    from dotenv import load_dotenv
    import os
    import requests
    import pandas as pd
    load_dotenv()
    client_id = os.getenv('CLIENT_ID')
    client_secret = os.getenv('CLIENT_SECRET')
    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret':client_secret
    }
    url = 'https://openapi.naver.com/v1/search/image'
    params = {'query':query, 'display':100 }
    response = requests.get(url, params=params, headers=headers)
    
    items = response.json()['items']
    # print(items[:2])
    items_list = []
    for item in items:
        items_list.append({
            '제목':item.get('title'),
            '링크':item.get('link'),
            '썸네일':item.get('thumbnail'),
            'sizeheight':item.get('sizeheight'),
            'sizewidth':item.get('sizewidth')
        })
    return pd.DataFrame(items_list)
get_image_list("젤라또")

,제목,링크,썸네일,sizeheight,sizewidth
0,오로시젤라또 하남점,https://ldb-phinf.pstatic.net/20241023_167/172...,https://search.pstatic.net/common/?type=b150&s...,1000,1000
1,#오늘클립챌린지 #젤라또 #커피없음,https://pup-post-phinf.pstatic.net/MjAyNjA2MDR...,https://search.pstatic.net/common/?type=b150&s...,1440,1080
2,#오늘클립챌린지 #마망젤라또 #성수젤라또,https://pup-post-phinf.pstatic.net/MjAyNjA2MTR...,https://search.pstatic.net/common/?type=b150&s...,1440,1080
3,#오늘클립챌린지 사누르에서 젤로 맛있늠 젤라또 #사누르 #발리여행,https://pup-post-phinf.pstatic.net/MjAyNjA0MjV...,https://search.pstatic.net/common/?type=b150&s...,1440,1080
4,#오늘클립챌린지 #순두부젤라또2호점,https://pup-post-phinf.pstatic.net/MjAyNjA0MTd...,https://search.pstatic.net/common/?type=b150&s...,1440,1080
...,...,...,...,...,...
95,"[카드뉴스] 사르르 녹아내리는 달콤함, 젤라또 맛집 BEST 5",http://imgnews.naver.net/image/028/2021/07/29/...,https://search.pstatic.net/common/?type=b150&s...,675,675
96,구워먹는 젤라또 레시피,https://recipe1.ezmember.co.kr/cache/recipe/20...,https://search.pstatic.net/sunny/?type=b150&sr...,1080,1080
97,"설악젤라또 - 강원, 속초시, 동명동 | 맛집검색 식신",https://img.siksinhot.com/place/17199799537260...,https://search.pstatic.net/sunny/?type=b150&sr...,1191,1080
98,이탈리아 아이스크림 젤라또 ‘구스띠모’ 온라인몰 본격 운영,http://imgnews.naver.net/image/003/2012/06/19/...,https://search.pstatic.net/common/?type=b150&s...,438,500


In [2]:
def get_image_list(query):
    'query로 검색한 이미지 정보(제목, 링크,썸네일, size) 100건 데이터 프레임을 return(방법2)'
    from dotenv import load_dotenv
    import os
    from urllib.request import urlopen, Request
    from urllib.parse import quote
    import json
    import pandas as pd
    load_dotenv()
    client_id = os.getenv('CLIENT_ID')
    client_secret = os.getenv('CLIENT_SECRET')
    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret':client_secret
    }
    query = quote(query)
    url = f'https://openapi.naver.com/v1/search/image?query={query}&display=100'
    request = Request(url, headers=headers)
    response = urlopen(request)
    
    items = json.loads(response.read())['items']
    # print(items[:2])
    items_list = []
    for item in items:
        items_list.append({
            '제목':item.get('title'),
            '링크':item.get('link'),
            '썸네일':item.get('thumbnail'),
            'sizeheight':item.get('sizeheight'),
            'sizewidth':item.get('sizewidth')
        })
    return pd.DataFrame(items_list)
df = get_image_list("젤라또")
df.sample()

,제목,링크,썸네일,sizeheight,sizewidth
83,로마에서 반드시 들러야 할 젤라또 맛집 3,http://imgnews.naver.net/image/5305/2023/08/08...,https://search.pstatic.net/common/?type=b150&s...,673,700


In [28]:
df.to_csv('data/img_list.csv', encoding='cp949', index=False)

UnicodeEncodeError: 'cp949' codec can't encode character '\U0001f495' in position 20: illegal multibyte sequence

In [ ]:
import pandas as pd
df = pd.read_csv('data/img_list.csv', encoding='cp949')

In [18]:
# df에 있는 이미지 로컬에 저장하기
print(df.loc[0, '링크'])
print(df.loc[0, '썸네일'])

https://ldb-phinf.pstatic.net/20241023_167/1729684454144OCqHx_JPEG/KakaoTalk_20240910_145227375.jpg
https://search.pstatic.net/common/?type=b150&src=https%3A%2F%2Fldb-phinf.pstatic.net%2F20241023_167%2F1729684454144OCqHx_JPEG%2FKakaoTalk_20240910_145227375.jpg


In [5]:
url = "https://a.net/a.jpg"
# '.' + url.split('.')[-1]
url[url.rfind('.'):]

'.jpg'

In [24]:
def save_image(attr, idx, link, query):
    'link의 이미지를 imge/attr_idx_query.확장자로 local에 저장'
    import requests, os
    from urllib.parse import urlparse
    response = requests.get(link)
    
    file_extension = link.split('.')[-1]
    
    #확장자 뒤에 ?가 있는 위치
    index = file_extension.find('?')
    if index != -1:
        file_extension = file_extension[:index]
    #확장자 뒤에 %가 있는 위치
    index = file_extension.find('%')
    if index != -1:
        file_extension = file_extension[:index]
    # 허용할 이미지 확장자인지
    valid_extension = ['jpg', 'jpeg', 'png', 'gif']
    if file_extension.lower() not in valid_extension:
        # 허용할 이미지 확장자가 아닌 경우(ex: .net)
        print(f'{idx}번째 {attr}의 url에서 확장자를 추출 못하여 header에서 도전해봄')
        content_type = response.headers.get('Content-Type', '')
        file_extension = content_type.split('imge/')[-1]
        file_extension = file_extension.replace('jpeg', 'jpg')
        file_extension = file_extension if file_extension in valid_extension else 'jpg'
    # image 폴더가 없으면 image 폴더 생성
    save_dir = 'image'
    os.makedirs(save_dir, exist_ok=True)
    
    # 이미지 저장
    with open(f'{save_dir}/{attr}_{idx:02}_{query}.{file_extension}', 'wb') as f:
        f.write(response.content) #response의 바이너리를 저장
        
save_image('썸네일', 0, df.loc[0, '썸네일'], '청바지')

In [ ]:
import pandas as pd
df = pd.read_csv('data/img_list.csv', encoding='cp949')

In [20]:
df.loc[0, '썸네일']

'https://search.pstatic.net/common/?type=b150&src=https%3A%2F%2Fldb-phinf.pstatic.net%2F20241023_167%2F1729684454144OCqHx_JPEG%2FKakaoTalk_20240910_145227375.jpg'

In [ ]:
# naver api 요청받아 이미지제목, link, 썸네일link 정보를 csv로 백업, 이미지link와 썸네일 link 이미지를 다운

In [27]:
def get_image_list_save_file(query):
    '''
    naver api 요청받아 이미지제목, link, 썸네일link 정보를 데이터프레임으로 return
    데이터프레임을 csv로 백업
    이미지link와 썸네일link 이미지를 다운(방법2)
    '''
    from dotenv import load_dotenv
    import os
    from urllib.request import urlopen, Request
    from urllib.parse import quote
    import json
    import pandas as pd
    load_dotenv()
    client_id = os.getenv('CLIENT_ID')
    client_secret = os.getenv('CLIENT_SECRET')
    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret':client_secret
    }
    query = quote(query)
    url = f'https://openapi.naver.com/v1/search/image?query={query}&display=100'
    request = Request(url, headers=headers)
    response = urlopen(request)
    
    items = json.loads(response.read())['items']
    items_list = []
    for idx, item in enumerate(items):
        link = item.get('link')
        thumbnail = item.get('thumbnail')
        items_list.append({
            '제목':item.get('title'),
            '링크':link,
            '썸네일':thumbnail,
            'sizeheight':item.get('sizeheight'),
            'sizewidth':item.get('sizewidth')
        })
        # 이미지 저장 save_image('메인', idx, linkm query) save
        save_image('메인', idx, link, query)
        save_image('썸네일', idx, thumbnail, query)
        if idx%20==0:
            print(f'==={idx}%진행완료===')
    result = pd.DataFrame(items_list)
    result.to_csv('image/img_list.csv', index=False)
    print('이미지 및 csv 저장완료')
    return result

In [28]:
df = get_image_list_save_file("청바지")

===0%진행완료===
===20%진행완료===
===40%진행완료===
===60%진행완료===
===80%진행완료===
이미지 및 csv 저장완료


## 2.3 XML 웹 데이터 수집
- RSS 서비스, openAPI사용
### 1) 전국 날씨 RSS를 BeautifulSoup를 이용한 xml 크롤링
- 기상청 RSS에서 1개월 기상정보를 서비스

In [43]:
import requests
# from urllib.request import urlopen
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
items_list = []
url = 'https://www.kma.go.kr/repositary/xml/fct/mon/img/fct_mon1rss_108_20260813.xml'
target = requests.get(url)
soup = BeautifulSoup(target.text, 'xml')
locals = soup.select('local_ta')
#print(locals[1])
for local in locals:
    local_name = local.select_one('local_ta_name').text.strip()
    week1_local_ta_normalYear = local.select_one('week1_local_ta_normalYear').text # 평년기온
    week1_local_ta_similarRange= local.select_one('week1_local_ta_similarRange').text # 예측범위
    week1_local_ta_minVal      = local.select_one('week1_local_ta_minVal').text # 평년보다 낮을 확률
    week1_local_ta_similarVal  = local.select_one('week1_local_ta_similarVal').text # 평년과 비슷할 확률
    week1_local_ta_maxVal      = local.select_one('week1_local_ta_maxVal').text # 평년보다 높을 확률
    items_list.append({
        '지역': local_name,
        '평년기온':week1_local_ta_normalYear,
        '예측범위':week1_local_ta_similarRange,
        '낮을확률':week1_local_ta_minVal,
        '같을확률':week1_local_ta_similarVal,
        '높을확률':week1_local_ta_maxVal,
    })
df = pd.DataFrame(items_list)
df.head()

,지역,평년기온,예측범위,낮을확률,같을확률,높을확률
0,"전국(제주도,북한제외)",23.7,23.1~24.3,10,30,60
1,서울ㆍ인천ㆍ경기도,23.9,23.3~24.5,10,30,60
2,강원도 영서,22.1,21.4~22.8,10,30,60
3,강원도 영동,22.0,21.3~22.7,10,30,60
4,대전ㆍ세종ㆍ충청남도,24.0,23.4~24.6,10,30,60


In [44]:
df['평년기온'] = df['평년기온'].astype(np.float64)
df['낮을확률'] = df['낮을확률'].astype(np.int16)
df['같을확률'] = df['같을확률'].astype(np.int16)
df['높을확률'] = df['높을확률'].astype('int')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   지역      13 non-null     object 
 1   평년기온    13 non-null     float64
 2   예측범위    13 non-null     object 
 3   낮을확률    13 non-null     int16  
 4   같을확률    13 non-null     int16  
 5   높을확률    13 non-null     int32  
dtypes: float64(1), int16(2), int32(1), object(2)
memory usage: 544.0+ bytes


### 3) xml로 응답하는 open APL활용
- data.go.kr에서
    * 서울특별시_버스위치정보조회 서비스(버스id, 버스 정류장 목록-정류장id와 정류장이름)
    * 서울특별시_노선정보조회 서비스(실시간 버스 위치 목록)

#### step1. 버스 번호의  busRouteId 받아오기
- 서울특별시_노선정보조회 서비스 - 3번 기능(getBusRouteList) 이용
- http://api.bus.go.kr/contents/sub02/getBusRouteList.html

In [52]:
# 인증키
from dotenv import load_dotenv
import os
load_dotenv()
key = os.getenv('KEY')

In [71]:
import requests
from urllib.request import quote
from bs4 import BeautifulSoup
from urllib.request import urlretrieve # xml을 로컬에 저장
# busNum = quote('마포01')
busNum = '600'
key = os.getenv('KEY')
url1 = f'http://ws.bus.go.kr/api/rest/busRouteInfo/getBusRouteList?ServiceKey={key}&strSrch={busNum}'
print(url1)
# 요청받은 xml을 백업
# savefilename1 = 'data/ch14_1.busInfo.xml'
# urlretrieve(url1, savefilename1)
# with open(savefilename1, encoding='utf-8') as f:
#     xml = f.read();
# soup = BeautifulSoup(xml, 'xml')
xml = requests.get(url1)
soup = BeautifulSoup(xml.text, 'xml')

http://ws.bus.go.kr/api/rest/busRouteInfo/getBusRouteList?ServiceKey=e193a220af0d32f65a1b9c832b64c8c6cd10c9ad2e81f8dd1951c40c7d6f1293&strSrch=600


In [72]:
for item in soup.select('itemList'):
    busRouteNm = item.select_one('busRouteNm').text
    if busNum == busRouteNm:
        busRouteId = item.find('busRouteId').text
        break
print('busRouteId = ', busRouteId)

busRouteId =  100100085


#### step2. 해당 busRouteId의  경유 정류장 목록 받아오기(정류장id, 정류장이름)
- 서울특별시_노선정보조회 서비스 - 4번 기능(getStaionsByRouteList) 이용
- http://api.bus.go.kr/contents/sub02/getStaionByRoute.html

In [77]:
import pandas as pd
url2 = f'http://ws.bus.go.kr/api/rest/busRouteInfo/getStaionByRoute?ServiceKey={key}&busRouteId={busRouteId}'
print(url2)
response = requests.get(url2)
soup = BeautifulSoup(response.text,'xml')
itemLists = soup.select('itemList')
print(f'{busNum}번 정류장 갯수 :', len(itemLists))
bus_station = []
for itemList in itemLists:
    stationNm = itemList.select_one('stationNm').text
    station = itemList.select_one('station').text
    gpsX = itemList.select_one('gpsX').text
    gpsY = itemList.select_one('gpsY').text
    bus_station.append([stationNm, station, gpsX, gpsY])
df_station = pd.DataFrame(bus_station, columns=['정류소명', 'id', '경도', '위도'])
df_station

http://ws.bus.go.kr/api/rest/busRouteInfo/getStaionByRoute?ServiceKey=e193a220af0d32f65a1b9c832b64c8c6cd10c9ad2e81f8dd1951c40c7d6f1293&busRouteId=100100085
600번 정류장 갯수 : 71


,정류소명,id,경도,위도
0,온수동종점,116000195,126.82122,37.491968
1,온수역,116000099,126.8244009171,37.492777021
2,우신중고등학교,116000097,126.8304506092,37.4930633974
3,궁동청소년문화의집.구로검사소,116000095,126.833633,37.493133
4,연세중앙교회,116000093,126.8354231175,37.4931082751
...,...,...,...,...
66,연세중앙교회,116000156,126.8359813615,37.4933027885
67,궁동청소년문화의집,116000094,126.832823,37.493319
68,우신중고등학교,116000096,126.830847,37.4933
69,온수역,116000098,126.8238226092,37.4928265566


In [78]:
pd.options.display.max_rows

60

#### step3. 해당 busRouteId의  차량들의 위치정보(차량번호, 혼잡도, 경도, 위도, 최종정류장id, 다음정류장id, 도착소요시간)
- 서울특별시_버스위치정보조회 서비스 - 2번 기능(getBusPosByRtidList) 이용
- http://api.bus.go.kr/contents/sub02/getStaionByRoute.html

In [108]:
url3 = f'http://ws.bus.go.kr/api/rest/buspos/getBusPosByRtid?serviceKey={Key}&busRouteId={busRouteId}'
print(url3)
response = requests.get(url3)
soup = BeautifulSoup(response.text, 'xml')
itemLists = soup.select('itemList')
print(f'{busNum}번 운행중인 버스는 {len(itemLists)}대입니다.')
bus_position = []
for itemList in itemLists:
    plainNo = itemList.select_one('plainNo').text
    congetion = itemList.select_one('congetion').text
    # 혼잡도(0 : 없음, 3 : 여유, 4 : 보통, 5 : 혼잡)
    congetion = '없음' if congetion=='0'\
            else '여유' if congetion=='3'\
            else '보통' if congetion=='4'\
            else '혼잡'
    gpsX = itemList.select_one('gpsX').text
    gpsY = itemList.select_one('gpsY').text
    lastStnId = itemList.select_one('lastStnId').text
    nextStId = itemList.select_one('nextStId').text
    nextStTm = itemList.select_one('nextStTm').text
    bus_position.append({
        '차량번호':plainNo,
        '혼잡도':congetion,
        '경도':gpsX,
        '위도':gpsY,
        '최종정류소id':lastStnId,
        '다음정류소id':nextStId,
        '도착소요시간':nextStTm
    })
df_position = pd.DataFrame(bus_position)
df_position.head(2)

http://ws.bus.go.kr/api/rest/buspos/getBusPosByRtid?serviceKey=e193a220af0d32f65a1b9c832b64c8c6cd10c9ad2e81f8dd1951c40c7d6f1293&busRouteId=100100085
600번 운행중인 버스는 20대입니다.


,차량번호,혼잡도,경도,위도,최종정류소id,다음정류소id,도착소요시간
0,서울70사6763,여유,126.838829,37.4938,116000093,116000446,304
1,서울71사3222,여유,126.849534,37.502618,116000125,116000004,988


In [109]:
df_station.loc[df_station['id']=='116000195', '정류소명'].iloc[0]

'온수동종점'

In [110]:
def get_station_name(row):
    row['최종정류장명'] = df_station.loc[df_station['id']==row['최종정류소id'], '정류소명'].iloc[0]
    row['다음정류장명'] = df_station.loc[df_station['id']==row['다음정류소id'], '정류소명'].iloc[0]
    return row

In [114]:
# get_station_name(df_position.iloc[0])
df_position = df_position.apply(get_station_name, axis=1)
df_position.head(1)

,차량번호,혼잡도,경도,위도,최종정류소id,다음정류소id,도착소요시간,최종정류장명,다음정류장명
0,서울70사6763,여유,126.838829,37.4938,116000093,116000446,304,연세중앙교회,동부골든.한신플러스타운


In [115]:
drop_col = df_position.columns.str.contains('id')
drop_column_names = df_position.columns[drop_col]
df_position.drop(drop_column_names, axis=1, inplace=True)

In [119]:
df_position['도착소요시간'] = round(df_position['도착소요시간'].astype('int')/60, 2)
df_position.head()

,차량번호,혼잡도,경도,위도,도착소요시간,최종정류장명,다음정류장명
0,서울70사6763,여유,126.838829,37.4938,5.07,연세중앙교회,동부골든.한신플러스타운
1,서울71사3222,여유,126.849534,37.502618,16.47,세곡초등학교,신도림동.구로역
2,서울70사6778,여유,126.863717,37.500321,9.02,고척중학교,신도림동.구로역
3,서울70사6760,여유,126.875772,37.500276,2.27,구일역.제일제당,신도림동.구로역
4,서울70사6680,여유,126.896963,37.512935,3.58,문래동남성아파트,영등포역


# Quiz
## 1. yes24의 베스트셀러 정보를 제공하는 사이트에서 베스트셀러 정보를 수집해서 파일에 저장하시오.
- 베스트셀러 정보 수집 주소 : http://www.yes24.com/24/category/bestseller
- YES24 베스트셀러 정보에서 순위1~48위까지의 "순위, 책이름, 저장, 출판사, 가격 정보를 출력하고, ch14_yes24_bestseller.csv로 백업

```
순위, 책이름, 저자, 출판사, 가격
1. 
2.
3. 
```

- 제출파일 : html소스파일과 ch14_yes24.csv

###  방법 1

In [127]:
# import
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
def collect(page):
    url = 'https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=24'
    params = {'p':page}
    response = requests.get(url, params=params)
    

print(url)

http://ws.bus.go.kr/api/rest/busRouteInfo/getBusRouteList?ServiceKey=e193a220af0d32f65a1b9c832b64c8c6cd10c9ad2e81f8dd1951c40c7d6f1293%strSrch=600


### 방법2